Import Libararies

In [66]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,roc_auc_score
from imblearn.over_sampling import SMOTE

In [67]:
df = pd.read_csv('/content/loan_prediction.csv')
print(df)

      Loan_ID  Gender Married  ... Credit_History Property_Area Loan_Status
0    LP001002    Male      No  ...            1.0         Urban           Y
1    LP001003    Male     Yes  ...            1.0         Rural           N
2    LP001005    Male     Yes  ...            1.0         Urban           Y
3    LP001006    Male     Yes  ...            1.0         Urban           Y
4    LP001008    Male      No  ...            1.0         Urban           Y
..        ...     ...     ...  ...            ...           ...         ...
609  LP002978  Female      No  ...            1.0         Rural           Y
610  LP002979    Male     Yes  ...            1.0         Rural           Y
611  LP002983    Male     Yes  ...            1.0         Urban           Y
612  LP002984    Male     Yes  ...            1.0         Urban           Y
613  LP002990  Female      No  ...            0.0     Semiurban           N

[614 rows x 13 columns]


In [68]:
X,y = df.drop("Loan_Status", axis=1),df["Loan_Status"]
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

**Pre-Processing Pipelines**

In [69]:
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler())])

categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),("encoder", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer([("num", numeric_pipeline, num_cols),("cat", categorical_pipeline, cat_cols)])

**Train-Test-split**

In [70]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

**Handle class Imbalance**

In [71]:
X_train_processed = preprocessor.fit_transform(X_train)

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_processed, y_train)

**Models**

In [72]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_resampled, y_resampled)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_resampled, y_resampled)

RandomForestClassifier(random_state=42)

**Evaluate Model**

In [73]:
pred = lr.predict(preprocessor.transform(X_test))
y_prob_lr = lr.predict_proba(X_test_processed)[:, 1]
roc_auc = roc_auc_score(y_test, y_prob_lr)
print("LogisticRegression:\n",classification_report(y_test, pred))
print("Roc Auc for Logistic Regression:\n",roc_auc)
pred = rf.predict(preprocessor.transform(X_test))
y_prob_rf = rf.predict_proba(X_test_processed)[:, 1]
roc_auc = roc_auc_score(y_test, y_prob_rf)
print("RandomForest:\n",classification_report(y_test, pred))
print("Roc Auc for Random Forest:\n",roc_auc)

LogisticRegression:
               precision    recall  f1-score   support

           N       0.89      0.66      0.76        38
           Y       0.86      0.96      0.91        85

    accuracy                           0.87       123
   macro avg       0.88      0.81      0.83       123
weighted avg       0.87      0.87      0.86       123

Roc Auc for Logistic Regression:
 0.8687306501547988
RandomForest:
               precision    recall  f1-score   support

           N       0.92      0.58      0.71        38
           Y       0.84      0.98      0.90        85

    accuracy                           0.85       123
   macro avg       0.88      0.78      0.81       123
weighted avg       0.86      0.85      0.84       123

Roc Auc for Random Forest:
 0.7846749226006192


In [74]:
importances = model.feature_importances_
feature_names = preprocessor.get_feature_names_out()

feat_imp = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print(feat_imp.head(10))

                          Feature  Importance
4             num__Credit_History    0.134706
0            num__ApplicantIncome    0.082233
2                 num__LoanAmount    0.073227
1          num__CoapplicantIncome    0.052174
509  cat__Property_Area_Semiurban    0.028890
3           num__Loan_Amount_Term    0.022521
508      cat__Property_Area_Rural    0.021974
500             cat__Dependents_0    0.017721
510      cat__Property_Area_Urban    0.016634
501             cat__Dependents_1    0.015417
